# **5. The price advisor**

Chapter 4 gave us a forecast. Chapter 5 turns it into advice a host can act on: when to
raise a price, when to leave it alone, and how to say so in an SMS.

One fact shapes everything in this chapter. **Our data contain no prices.** The synthetic
occupancy was generated without any price response, so we cannot learn or test how guests
react to price. Every claim about revenue rests on a stated assumption, which we vary
instead of defending. The advisor therefore has to be four things: honest about that
assumption, safe when the assumption is wrong, easy to explain, and able to learn from
what hosts accept or reject.

## **5.1 When is a price change safe? A revenue grid**

### **5.1.1 Why**
Before writing any rule, we find out when raising or cutting a price can lose money, and
how that depends on two things we do not know: how sensitive guests are to price, and
how much demand exceeds the rooms.

### **5.1.2 Model**
- **D** is the demand at the current price, as a share of rooms. It can exceed the
  occupancy cap of 98%, meaning more guests would have come than there are rooms.
- A price change of **r** (for example +10%) changes demand to D x (1 + r)^eps, where
  **eps** is the price elasticity, a negative number. eps = -0.6 means guests barely react;
  eps = -2.0 means they react strongly.
- Occupancy is demand, limited to the cap. Revenue is price times occupancy, so the ratio
  of revenue after to revenue before is (1 + r) x occupancy after / occupancy before.
- We do not claim any eps is true. We scan from -0.6 to -2.0.

### **5.1.3 Method**
1. Compute the revenue change on a grid of D (0.60 to 1.40), eps (-0.6, -1.0, -1.4, -2.0)
   and price changes of +5%, +10%, +20% and -10%.
2. Derive the largest safe raise: for D at or above the cap and |eps| above 1, a raise r is
   safe (revenue does not fall) up to r* = (D / cap)^(1 / (|eps| - 1)) - 1. For eps = -2
   this simplifies to D / cap - 1. For D below the cap, no raise is safe when |eps| is
   above 1. Check the formula against a numerical search.

### **5.1.4 Predictions (written before the run)**
1. For D = 0.80 or 0.90, a +10% price rise changes revenue by +3.9%, 0.0%, -3.7% and
   -9.1% for eps = -0.6, -1.0, -1.4 and -2.0.
2. For D = 1.20, a +20% rise changes revenue by +20.0%, +20.0%, +13.8% and +2.0% for the
   same four elasticities.
3. The numerical largest safe raise matches the formula within 0.1 percentage points at
   eps = -2.0 and eps = -1.4.
4. A -10% price cut loses exactly 10.0% for every eps when D is 1.00 or more (the cap
   binds). At D = 0.60 it changes revenue by -4.1%, 0.0%, +4.3% and +11.1%.

### **5.1.5 Decision rules (if the predictions hold)**
- **R1.** Never suggest a price cut by default. A cut helps only if guests are very price
  sensitive and rooms are not close to full, and we cannot know either.
- **R2.** Suggest a raise only where demand is expected to exceed the rooms, and cap it
  at the safe limit for eps = -2, which is D / cap - 1. That raise cannot lose revenue
  for any elasticity up to magnitude 2.
- **R3.** On quiet days suggest non-price actions instead (a promotion, an airtime reward
  for bringing a friend), which the hackathon brief also mentions.
- **R4.** To apply R2 we need an estimate of D, and our forecast is capped at 98%. Step
  5.2 tests whether the model's unclipped prediction can estimate demand above capacity,
  using the generator's known latent demand.

In [1]:
import numpy as np
import pandas as pd

CAP = 0.98


def revenue_ratio(D, r, eps, cap=CAP):
    """Revenue after a price change of r, divided by revenue at the current price.
    D: demand at the current price as a share of rooms (it can exceed the cap).
    eps: price elasticity of demand (negative). Occupancy can never exceed cap."""
    occ_now = np.minimum(D, cap)
    occ_new = np.minimum(D * (1 + r) ** eps, cap)
    return (1 + r) * occ_new / occ_now


D_grid = np.array([0.60, 0.80, 0.90, 0.95, 1.00, 1.05, 1.10, 1.20, 1.40])
eps_grid = np.array([-0.6, -1.0, -1.4, -2.0])

for r in [0.05, 0.10, 0.20, -0.10]:
    grid = pd.DataFrame(
        100 * (revenue_ratio(D_grid[:, None], r, eps_grid[None, :]) - 1),
        index=[f"D={d:.2f}" for d in D_grid],
        columns=[f"eps={e}" for e in eps_grid],
    )
    print(f"Revenue change (%) for a price change of {r:+.0%}:")
    print(grid.round(1).to_string())
    print()


def breakeven_numeric(D, eps, cap=CAP, step=0.0005, rmax=1.5):
    """Largest raise r for which revenue does not fall, found by search."""
    rs = np.arange(0.0, rmax + step, step)
    ok = revenue_ratio(D, rs, eps, cap) >= 1 - 1e-12
    if ok.all():
        return rmax
    first_bad = int(np.argmax(~ok))
    return float(rs[first_bad - 1]) if first_bad > 0 else 0.0


def breakeven_formula(D, eps, cap=CAP):
    if D < cap:
        return 0.0
    return (D / cap) ** (1 / (abs(eps) - 1)) - 1


rows = []
for D in [0.80, 0.90, 0.98, 1.00, 1.05, 1.10, 1.20, 1.30]:
    rows.append({
        "D": D,
        "eps -2.0 numeric": breakeven_numeric(D, -2.0),
        "eps -2.0 formula": breakeven_formula(D, -2.0),
        "eps -1.4 numeric": breakeven_numeric(D, -1.4),
        "eps -1.4 formula": breakeven_formula(D, -1.4),
    })
be = pd.DataFrame(rows).set_index("D")
print("Largest safe price rise (%), numeric search against the formula:")
print((100 * be).round(1).to_string())
for e in ("-2.0", "-1.4"):
    gap = (100 * (be[f"eps {e} numeric"] - be[f"eps {e} formula"])).abs().max()
    print(f"largest gap at eps {e}: {gap:.2f} percentage points")

Revenue change (%) for a price change of +5%:
        eps=-0.6  eps=-1.0  eps=-1.4  eps=-2.0
D=0.60       2.0       0.0      -1.9      -4.8
D=0.80       2.0       0.0      -1.9      -4.8
D=0.90       2.0       0.0      -1.9      -4.8
D=0.95       2.0       0.0      -1.9      -4.8
D=1.00       4.1       2.0       0.1      -2.8
D=1.05       5.0       5.0       5.0       2.0
D=1.10       5.0       5.0       5.0       5.0
D=1.20       5.0       5.0       5.0       5.0
D=1.40       5.0       5.0       5.0       5.0

Revenue change (%) for a price change of +10%:
        eps=-0.6  eps=-1.0  eps=-1.4  eps=-2.0
D=0.60       3.9       0.0      -3.7      -9.1
D=0.80       3.9       0.0      -3.7      -9.1
D=0.90       3.9       0.0      -3.7      -9.1
D=0.95       3.9       0.0      -3.7      -9.1
D=1.00       6.0       2.0      -1.8      -7.2
D=1.05      10.0       7.1       3.1      -2.6
D=1.10      10.0      10.0       8.0       2.0
D=1.20      10.0      10.0      10.0      10.0
D=1.40      1

## **5.2 Review of results: the revenue grid (step 5.1)**

### **5.2.1 Verdict**
All four predictions held exactly, so the grid supports decision rules R1 to R3. It also
showed something I had not planned: the break-even raise (D / cap - 1) gains nothing in the
worst case, so the advisor should suggest a smaller raise inside it. Rule R4 (can we
estimate demand above the rooms?) is still open and is the subject of step 5.3.

### **5.2.2 Prediction scorecard**

| Prediction (5.1.4) | Result | Verdict |
|---|---|---|
| D = 0.80 or 0.90, +10%: +3.9, 0.0, -3.7, -9.1 | Exactly those values in both rows | Held |
| D = 1.20, +20%: +20.0, +20.0, +13.8, +2.0 | Exactly those values | Held |
| Numeric safe raise matches the formula within 0.1 points | Largest gaps 0.05 (eps -2.0) and 0.03 (eps -1.4) | Held |
| A -10% cut loses exactly 10.0% for D of 1.00 or more; at D = 0.60 it gives -4.1, 0.0, +4.3, +11.1 | Exactly those values | Held |

### **5.2.3 What the numbers show**

#### **5.2.3.1 Below capacity, a raise is a bet on insensitive guests**
For every D from 0.60 to 0.95 the result is the same, because occupancy is not limited by
the rooms. A +5% raise gives +2.0%, 0.0%, -1.9% and -4.8% at eps = -0.6, -1.0, -1.4 and
-2.0; +10% gives +3.9%, 0.0%, -3.7%, -9.1%; +20% gives +7.6%, 0.0%, -7.0%, -16.7%. The sign
flips at eps = -1: a raise helps only if guests react less than proportionally.

#### **5.2.3.2 Above capacity, a raise is safe up to the excess**

| D (demand as a share of rooms) | Largest safe raise at eps = -2.0 | at eps = -1.4 |
|---|---|---|
| 1.00 | 2.0% | 5.2% |
| 1.05 | 7.1% | 18.8% |
| 1.10 | 12.2% | 33.4% |
| 1.20 | 22.4% | 65.9% |
| 1.30 | 32.6% | 102.6% |

At eps = -2 the limit is exactly D / cap - 1, and the formula agrees with the numerical
search to within 0.05 points. The limit is much larger for milder elasticities.

#### **5.2.3.3 Cuts help only in a narrow zone**
At D = 0.60 a -10% cut gains +4.3% (eps = -1.4) and +11.1% (eps = -2.0); at D = 0.80 it
gains +4.3% and +10.3%. By D = 0.90 the cap binds after the cut and the cut loses 2.0% for
every eps of -1.0 or more negative (-4.1% at eps = -0.6). At D = 0.95 it loses 7.2% for every
eps, and at D of 1.00 or more it loses exactly 10%. A cut pays only for very sensitive
guests and well below capacity, and we cannot tell either from our data.

#### **5.2.3.4 The gamble zone**
For D between about 0.90 and 1.00, neither move is safe: raises lose if guests are
sensitive (eps below -1), and cuts lose because the rooms fill anyway. The advisor holds
there.

#### **5.2.3.5 A refinement: how big should a raise be?**
The formula D / cap - 1 is the break-even raise, which gains nothing at eps = -2, so it is
too aggressive to suggest. Three natural sizes (computed from the formulas):

| D | Keeps rooms full at eps = -2: (D/cap)^(1/2) - 1 | Best at eps = -1.4: (D/cap)^(1/1.4) - 1 | Break-even at eps = -2: D/cap - 1 |
|---|---|---|---|
| 1.10 | 5.9% | 8.6% | 12.2% |
| 1.20 | 10.7% | 15.6% | 22.4% |

I choose the middle one, **(D/cap)^(1/1.4) - 1, rounded down to a multiple of 5%**, with a
minimum of 5% and a maximum of 25%. It is never above the break-even at eps = -2, so
it loses nothing for any elasticity up to magnitude 2. The elasticity of -1.4 is the middle
of the scanned range, an assumption and not an estimate. A 5% suggestion needs D of about
1.05 or more.

### **5.2.4 Limits**
- Constant elasticity is a simplification, and the range -2.0 to -0.6 is assumed.
- The model counts revenue only. It ignores costs per occupied room, competitors and
  guests' memory of past prices.
- In practice D is unknown. We have only a forecast capped at the rooms available.

### **5.2.5 Decision**
Adopt R1 (no default cuts), R2 (raise only where demand exceeds rooms, sized by the
refined formula) and R3 (non-price actions on quiet days). Test R4 next.

## **5.3 Next step: can we see demand above the rooms?**

### **5.3.1 Why**
R2 needs D, demand as a share of rooms, which can exceed 100%. Our occupancy forecast stops
at the rooms available, and the model was trained on occupancy that is itself capped, so
it never saw demand above the cap. We test whether the model's **unclipped** prediction is
a usable estimate of D, using the generator's own latent demand as the truth.

### **5.3.2 Method**
1. Add unclipped columns to the forecaster: `demand`, `demand_low`, `demand_high` (Part A).
2. Refit on all history and forecast 146 days (to 31 March 2027) for each destination.
3. Compare the unclipped prediction (D_hat) with the generator's noise-free latent demand
   (D_true), on days where D_true exceeds the 98% cap.
4. Apply the advisor's raise rule to D_hat (rule A) and to the lower bound of the demand
   interval (rule B), and judge each against what D_true says is safe.

### **5.3.3 The raise rule used**
For D above the cap, raise by (D / cap)^(1 / 1.4) - 1, rounded down to a multiple of 5%,
with a minimum of 5% and a maximum of 25%. Otherwise no raise.

### **5.3.4 Predictions (written before the run)**
1. Days in the 146 where the true demand is above the cap: Naivasha 3 to 6, Diani 5 to 14,
   Lamu 3 to 12, Maasai Mara 0 to 4, Amboseli 0 to 2.
2. On those days D_hat is below D_true: the ratio D_hat / D_true is between 0.85 and 0.97 in
   every destination that has such days.
3. Rule A is safe: at least 95% of its suggestions are at or below the break-even computed
   from the true demand at eps = -2, and the mean revenue change on suggested days is not
   negative at eps = -2.0 and eps = -1.4.
4. Rule A is cautious to the point of silence: overall it finds fewer than half of the
   opportunity days, meaning the days where the same rule applied to the true demand would
   suggest a raise.
5. Rule B suggests a raise on no more days than rule A, and is also at least 95% safe.

### **5.3.5 Decision rule**
- If rule A is safe but finds fewer than half of the opportunities, test a censoring-aware
  fit in 5.4 (observations at the cap treated as "at least this much", not as exact).
- If rule A finds half or more and is safe, adopt it, and keep rule B as a conservative
  option.
- If rule A is not safe (under 95%), adopt rule B and find out why D_hat is too high.

In [2]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../src").resolve()))
import locat.forecasting.features as ffeat
import locat.forecasting.model as fmodel
import locat.forecasting.synthetic as synth

CAP = 0.98
out_dir = Path("../data/processed")
DESTINATIONS = ["Maasai Mara", "Naivasha", "Diani", "Lamu", "Amboseli"]
HISTORY_END = pd.Timestamp("2026-11-05")
FUTURE_END = pd.Timestamp("2027-03-31")


def revenue_ratio(D, r, eps, cap=CAP):
    occ_now = np.minimum(D, cap)
    occ_new = np.minimum(D * (1 + r) ** eps, cap)
    return (1 + r) * occ_new / occ_now


def suggested_raise(D, cap=CAP, eps_mid=-1.4, step=0.05, min_raise=0.05, max_raise=0.25):
    """Best raise at the middle elasticity; never above the break-even at eps = -2."""
    D = np.asarray(D, dtype=float)
    r = np.where(D > cap, (D / cap) ** (1 / abs(eps_mid)) - 1, 0.0)
    r = np.floor(r / step + 1e-9) * step
    return np.where(r < min_raise, 0.0, np.minimum(r, max_raise))


# ---- setup: data, forecaster, truth ----
cal = pd.read_csv(out_dir / "calendar_daily.csv", parse_dates=["date"])
seasonal = pd.read_csv(out_dir / "seasonal_index.csv")
idx_map = dict(zip(seasonal["month_number"], seasonal["seasonal_index"]))
feats = ffeat.build_features(cal, idx_map)
q = pd.read_csv(out_dir / "forecast_interval_quantiles.csv", index_col=0)
syn = pd.read_csv(out_dir / "synthetic_occupancy.csv", parse_dates=["date"])
occ = syn.pivot(index="date", columns="destination", values="occupancy")[DESTINATIONS]

assump = json.loads((out_dir / "synthetic_assumptions.json").read_text())
params = pd.DataFrame(assump["params"])
month_adjust = pd.DataFrame.from_dict(assump["month_adjust"], orient="index")
settings = assump["globals"]

fc = fmodel.OccupancyForecaster(feats, q).fit(occ)
truth = synth.generate_history(
    cal, params, month_adjust, dict(settings, noise_sd=0.0), idx_map, FUTURE_END,
    normalise_until=HISTORY_END,
)
truth_latent = truth.pivot(index="date", columns="destination", values="latent_demand")[DESTINATIONS]

# ---- forecasts of demand, side by side with the truth ----
horizon = (FUTURE_END - fc.last_date).days
parts = []
for dest in DESTINATIONS:
    f = fc.forecast(dest, horizon)
    assert np.allclose(f["forecast"], np.clip(f["demand"], 0.01, CAP))  # old columns unchanged
    parts.append(pd.DataFrame({
        "destination": dest, "date": f.index,
        "D_hat": f["demand"].to_numpy(), "D_low": f["demand_low"].to_numpy(),
        "D_true": truth_latent.loc[f.index, dest].to_numpy(),
    }))
d = pd.concat(parts, ignore_index=True)
d["excess"] = d["D_true"] > CAP
print(f"compared {horizon} days per destination: {d['date'].min().date()} to {d['date'].max().date()}\n")

# 1) Days with true demand above the rooms, and how well D_hat sees them
ex = d[d["excess"]].groupby("destination").agg(
    days=("D_true", "size"),
    mean_D_true=("D_true", "mean"),
    mean_D_hat=("D_hat", "mean"),
    share_D_hat_above_cap=("D_hat", lambda s: (s > CAP).mean()),
).reindex(DESTINATIONS)
ex["days"] = ex["days"].fillna(0).astype(int)
ex["D_hat_over_D_true"] = ex["mean_D_hat"] / ex["mean_D_true"]
print("Days where true demand exceeds the rooms:")
print(ex.round(2).to_string())

# 2) The raise rule applied to D_hat (rule A) and to its lower bound (rule B)
EPS = [-2.0, -1.4, -1.0]
d["r_true_rule"] = suggested_raise(d["D_true"])            # the rule with perfect knowledge
d["breakeven_true"] = np.maximum(d["D_true"] / CAP - 1, 0.0)
rows = []
for rule, col in [("A_point", "D_hat"), ("B_lower_bound", "D_low")]:
    d[f"r_{rule}"] = suggested_raise(d[col])
    for dest in DESTINATIONS + ["ALL"]:
        s = d if dest == "ALL" else d[d["destination"] == dest]
        sugg = s[f"r_{rule}"] > 0
        opp = s["r_true_rule"] > 0
        row = {
            "rule": rule, "destination": dest,
            "suggested_days": int(sugg.sum()),
            "opportunity_days": int(opp.sum()),
            "hit_days": int((sugg & opp).sum()),
            "safe_share_%": (100 * float((s.loc[sugg, f"r_{rule}"] <= s.loc[sugg, "breakeven_true"] + 1e-9).mean())
                             if sugg.any() else np.nan),
        }
        for e in EPS:
            chg = 100 * (revenue_ratio(s.loc[sugg, "D_true"].to_numpy(),
                                       s.loc[sugg, f"r_{rule}"].to_numpy(), e) - 1)
            row[f"rev_change_eps{e}_%"] = chg.mean() if sugg.any() else np.nan
        rows.append(row)
print("\nRaise rule judged against the true demand:")
print(pd.DataFrame(rows).round(1).to_string(index=False))

# 3) A concrete look at the festive days
def show(dest, start, end):
    s = d[(d["destination"] == dest) & (d["date"] >= start) & (d["date"] <= end)]
    print(f"\n{dest}, {start} to {end}:")
    print(pd.DataFrame({
        "day": s["date"].dt.strftime("%a %d %b").to_numpy(),
        "D_true": s["D_true"].round(2).to_numpy(), "D_hat": s["D_hat"].round(2).to_numpy(),
        "D_low": s["D_low"].round(2).to_numpy(),
        "raise_if_known_%": (100 * s["r_true_rule"]).round(0).astype(int).to_numpy(),
        "raise_A_%": (100 * s["r_A_point"]).round(0).astype(int).to_numpy(),
        "raise_B_%": (100 * s["r_B_lower_bound"]).round(0).astype(int).to_numpy(),
    }).to_string(index=False))

show("Naivasha", "2026-12-24", "2026-12-28")
show("Naivasha", "2026-12-31", "2027-01-03")
show("Diani", "2026-12-22", "2026-12-31")

compared 146 days per destination: 2026-11-06 to 2027-03-31

Days where true demand exceeds the rooms:
             days  mean_D_true  mean_D_hat  share_D_hat_above_cap  D_hat_over_D_true
destination                                                                         
Maasai Mara     0          NaN         NaN                    NaN                NaN
Naivasha        4         1.06        0.95                    0.0               0.90
Diani           4         1.10        1.03                    1.0               0.94
Lamu            0          NaN         NaN                    NaN                NaN
Amboseli        0          NaN         NaN                    NaN                NaN

Raise rule judged against the true demand:
         rule destination  suggested_days  opportunity_days  hit_days  safe_share_%  rev_change_eps-2.0_%  rev_change_eps-1.4_%  rev_change_eps-1.0_%
      A_point Maasai Mara               0                 0         0           NaN                   NaN   

## **5.4 Review of results: can we see demand above the rooms? (step 5.3)**

### **5.4.1 Verdict**
The model's unclipped prediction is a usable but low estimate of demand above the rooms: it
sits at 90% to 94% of the truth on the days that matter. The raise rule applied to it
suggested nothing, in either variant. The larger finding is that even perfect knowledge
supports a raise on only 6 of 730 destination-days in this window, because demand in our
synthetic world rarely exceeds the rooms and then only by 3% to 13%. That changes what the
advisor should be, and it changes the next step. Of the five predictions, two held outright,
one held on part of its range, and two held only because nothing was suggested.

### **5.4.2 Prediction scorecard**

| Prediction (5.3.4) | Result | Verdict |
|---|---|---|
| Days with true demand above the cap: Naivasha 3-6, Diani 5-14, Lamu 3-12, Maasai Mara 0-4, Amboseli 0-2 | Naivasha 4, Diani 4, Lamu 0, Maasai Mara 0, Amboseli 0 | 3 of 5 in range; Diani and Lamu below it |
| D_hat / D_true between 0.85 and 0.97 on those days | Naivasha 0.90, Diani 0.94 | Held |
| Rule A is at least 95% safe, with non-negative revenue change | No suggestions, so safety cannot be measured | Vacuously true; untested |
| Rule A finds fewer than half of the opportunity days | 0 of 6 | Held |
| Rule B suggests no more days than A and is at least 95% safe | 0 days, equal to A | Vacuously true; untested |

### **5.4.3 What the numbers show**

#### **5.4.3.1 Why the count prediction missed**
In the earlier chart, Diani's truth looked flat at about 98% over many days. Those days are
**capped**, not in excess: true demand on 27 December is 0.97, just under the cap. Demand
above the rooms is rare: 4 days in each of Naivasha and Diani, none in the other three
destinations. I misread a plateau as an excess.

#### **5.4.3.2 The unclipped prediction: a low but informative estimate**

| Destination | Days of excess demand | Mean true D | Mean D_hat | D_hat / D_true | Share of those days with D_hat above the cap |
|---|---|---|---|---|---|
| Naivasha | 4 | 1.06 | 0.95 | 0.90 | 0% |
| Diani | 4 | 1.10 | 1.03 | 0.94 | 100% |

The model underestimates by 6% to 10%, as predicted. Two reasons are plausible and neither is
tested: the training data show occupancy capped at 98%, so peak days look lower than they
were; and the model combines separately estimated effects, which can drift at their
extreme combinations. The sample is 8 destination-days, so the ratios are rough.

#### **5.4.3.3 Why the rule stayed silent**

| Day | D_true | D_hat | D_low | Rule on the truth | Rule A | Rule B |
|---|---|---|---|---|---|---|
| Naivasha, Fri 25 Dec | 1.09 | 0.96 | 0.85 | +5% | none | none |
| Diani, Fri 25 Dec | 1.13 | 1.04 | 0.85 | +10% | none | none |
| Naivasha, Fri 1 Jan | 1.03 | 0.95 | 0.83 | none | none | none |

Three separate things stack up. First, D_hat is 6% to 10% low, so in Naivasha it falls
below the cap (0.96 is under 0.98) and no raise is possible. Second, the rule rounds down
to multiples of 5% and requires at least 5%: in Diani the raise from D_hat is 4.4%, which
rounds to nothing, while the same rule on the truth gives 10%. Third, the lower bound is lower
still (0.85), so rule B cannot fire at all. On 1 January in Naivasha the truth itself
supports only 3.6%, below the 5% minimum.

#### **5.4.3.4 How rare the opportunity is**
Applied to the true demand, the rule suggests a raise on 6 days across all destinations:
Naivasha on 25 and 26 December (+5%) and four days in Diani (+10% on 25 and 26 December). In 146 days, across five
destinations, that is under 1% of destination-days. The reason is in our own assumptions: the
generator's peak boosts are moderate and the cap is 98%, so true demand never exceeds the
rooms by more than 13%. Real destinations may differ, but we cannot assume it.

### **5.4.4 Limits**
- 146 days and 6 opportunity days make a tiny sample. The conclusions about this window are
  solid; the 0.90 and 0.94 ratios are rough.
- Everything here is about the safe raise at the middle elasticity. We still know nothing
  about real guests' price sensitivity.

### **5.4.5 Decision**
By the letter of the rule I wrote in 5.3.5, this result sends us to a censoring-aware fit.
I recommend a different next step, because the whole prize for fixing the estimate is those
6 days. I would first measure that prize (step 5.5): the value of perfect knowledge for
confident raises, set against two other policies. If the prize is small, the censoring-aware
fit is skipped and the advisor is designed around what is actually valuable. This departs
from the pre-registered rule, and I am saying so explicitly; if you prefer to follow the rule
as written, say so and we do the fit first.

## **5.5 Next step: what is perfect knowledge worth, and what should the advisor do?**

### **5.5.1 Why**
Before spending effort on a better demand estimate, we measure the most it could be worth.
We also compare the confident raise with two other behaviours: a small experiment on busy
nights and a blanket weekend premium, which many hosts apply by habit.

### **5.5.2 Method**
Over the 146-day window, five destinations, with the true demand as the real world and the
price at the current level equal to 1:
1. **Hold:** revenue is occupancy at the current price (capped at 98%).
2. **P1, confident raise with perfect knowledge:** the raise rule applied to the true demand.
3. **P2, confident raise on the model's estimate:** rule A from 5.3 (it suggests nothing).
4. **P3, test +5% on busy nights:** +5% on days where D_hat is at or above a threshold T
   (0.80, 0.85, 0.90 or 0.95) and rule A has not already suggested a raise.
5. **P4, a blanket weekend premium:** +5% on every Friday and Saturday night.
Each policy is scored as the percentage change in total revenue against holding, at four
elasticities: -2.0, -1.4, -1.0 and -0.6.

### **5.5.3 Predictions (written before the run)**
1. P1 changes total revenue by **+0.05% to +0.12%** at every elasticity.
2. P2 changes it by exactly 0.00% (no suggestions).
3. P3 at T = 0.85 touches **12 to 40 days** and changes revenue by **-0.40% to -0.15%** at
   eps = -2.0, and by **+0.05% to +0.20%** at eps = -0.6.
4. P4 touches about 28% of destination-days and changes revenue by **-2.2% to -1.4%** at
   eps = -2.0 and **+0.5% to +1.0%** at eps = -0.6.
5. P4's loss at eps = -2.0 is at least **4 times** P3's loss at T = 0.85.

### **5.5.4 Decision rules**
- If P1's gain is below 0.5% of revenue at every elasticity, **skip the censoring-aware fit**
  and record why.
- The **experiment tier** (P3) is included in the advisor only if its worst case (eps = -2.0)
  is no worse than -0.5% of window revenue; choose the lowest T among 0.80, 0.85, 0.90 and
  0.95 that meets this, so we test on as many nights as the bound allows.
- A blanket weekend premium (P4) is never offered as advice unless it meets the same bound.

In [3]:
# ---- add the weekend flag to the comparison table from step 5.3 ----
d = d.drop(columns=["fri_sat_night"], errors="ignore").merge(
    feats[["fri_sat_night"]].reset_index(), on="date", how="left"
)
d["hold_rev"] = np.minimum(d["D_true"], CAP)   # occupancy at the current price (price = 1)
EPS_ALL = [-2.0, -1.4, -1.0, -0.6]
base = d["hold_rev"].sum()
n_rows = len(d)


def policy_effect(name, r):
    """Revenue change (% of window revenue) for a price change r on each row."""
    r = np.asarray(r, dtype=float)
    on = r != 0
    row = {"policy": name, "days": int(on.sum()), "share_of_days_%": 100 * on.mean()}
    for e in EPS_ALL:
        new = d["hold_rev"].to_numpy().copy()
        new[on] = new[on] * revenue_ratio(d["D_true"].to_numpy()[on], r[on], e)
        row[f"eps {e}"] = 100 * (new.sum() - base) / base
    return row


rows = [
    policy_effect("P1 confident raise, perfect knowledge", d["r_true_rule"]),
    policy_effect("P2 confident raise on model estimate (rule A)", d["r_A_point"]),
]
for T in [0.80, 0.85, 0.90, 0.95]:
    r = np.where((d["D_hat"] >= T) & (d["r_A_point"] == 0), 0.05, d["r_A_point"])
    rows.append(policy_effect(f"P3 test +5% when demand >= {T:.0%}", r))
rows.append(policy_effect("P4 blanket +5% on every Fri/Sat night",
                          np.where(d["fri_sat_night"] == 1, 0.05, 0.0)))

print(f"window: {n_rows} destination-days, hold revenue = {base:.1f} room-nights at price 1\n")
print("Revenue change against holding the price (% of window revenue):")
print(pd.DataFrame(rows).round(2).to_string(index=False))

busy = d[d["D_hat"] >= 0.85]
print("\nDays with model demand >= 85%, by destination:")
print(busy.groupby("destination").agg(
    days=("D_hat", "size"),
    fri_sat_nights=("fri_sat_night", "sum"),
    mean_D_true=("D_true", "mean"),
).reindex(DESTINATIONS).fillna(0).round(2).to_string())

p4 = next(r for r in rows if r["policy"].startswith("P4"))
p3 = next(r for r in rows if r["policy"] == "P3 test +5% when demand >= 85%")
print(f"\nP4 loss at eps -2.0 is {abs(p4['eps -2.0']) / abs(p3['eps -2.0']):.1f} times P3 (T = 85%)")

window: 730 destination-days, hold revenue = 413.6 room-nights at price 1

Revenue change against holding the price (% of window revenue):
                                       policy  days  share_of_days_%  eps -2.0  eps -1.4  eps -1.0  eps -0.6
        P1 confident raise, perfect knowledge     6             0.82      0.07      0.09      0.09      0.09
P2 confident raise on model estimate (rule A)     0             0.00      0.00      0.00      0.00      0.00
               P3 test +5% when demand >= 80%    20             2.74     -0.05      0.04      0.09      0.15
               P3 test +5% when demand >= 85%    14             1.92      0.01      0.06      0.09      0.12
               P3 test +5% when demand >= 90%    10             1.37      0.05      0.08      0.09      0.10
               P3 test +5% when demand >= 95%     6             0.82      0.07      0.07      0.07      0.07
        P4 blanket +5% on every Fri/Sat night   210            28.77     -1.38     -0.50      0.09

## **5.6 Review of results: what perfect knowledge is worth (step 5.5)**

### **5.6.1 Verdict**
Perfect knowledge of demand would add only 0.07% to 0.09% to revenue through confident
raises, so the censoring-aware fit is skipped. The small test tier passes its safety bound
and is adopted. The blanket weekend premium fails it and is never offered. The larger
finding is that in our synthetic world price changes are worth at most 0.15% of revenue,
so the advisor's value lies mainly in the demand outlook and in guardrails against
costly habits. Of the five predictions, two held, two held with a caveat, and one half held.

### **5.6.2 Prediction scorecard**

| Prediction (5.5.3) | Result | Verdict |
|---|---|---|
| P1 changes revenue by +0.05% to +0.12% at every elasticity | +0.07, +0.09, +0.09, +0.09 | Held |
| P2 changes revenue by exactly 0.00% | 0 days, 0.00% | Held |
| P3 at T = 0.85 touches 12 to 40 days, -0.40% to -0.15% at eps -2.0, +0.05% to +0.20% at eps -0.6 | 14 days; +0.01% at eps -2.0; +0.12% at eps -0.6 | Half held: days and the upside held; the downside was predicted too pessimistically |
| P4 touches about 28% of days; -2.2% to -1.4% at eps -2.0, +0.5% to +1.0% at eps -0.6 | 28.8% of days; -1.38%; +0.70% | Held (-1.38 is 0.02 outside the edge) |
| P4's loss is at least 4 times P3's loss | The printed "174 times" divides a loss by a gain | Held in substance (P4 loses, P3 does not); the ratio itself is meaningless |

### **5.6.3 What the numbers show**

#### **5.6.3.1 Confident raises: little to gain from knowing more**

| Policy | Days | Share of days | eps -2.0 | eps -1.4 | eps -1.0 | eps -0.6 |
|---|---|---|---|---|---|---|
| P1 raise, perfect knowledge | 6 | 0.82% | +0.07% | +0.09% | +0.09% | +0.09% |
| P2 raise on the model's estimate | 0 | 0.00% | 0.00% | 0.00% | 0.00% | 0.00% |

The prize for a perfect estimate is about one part in a thousand of revenue. Decision rule
(a) says to skip the censoring-aware fit when this is below 0.5%, and it is far below. I
record that we did not do it, and why.

#### **5.6.3.2 The test tier: safer than I predicted**

| Threshold T | Days | eps -2.0 | eps -1.4 | eps -1.0 | eps -0.6 |
|---|---|---|---|---|---|
| 80% | 20 | -0.05% | +0.04% | +0.09% | +0.15% |
| 85% | 14 | +0.01% | +0.06% | +0.09% | +0.12% |
| 90% | 10 | +0.05% | +0.08% | +0.09% | +0.10% |
| 95% | 6 | +0.07% | +0.07% | +0.07% | +0.07% |

I predicted losses at eps = -2 because I assumed the flagged nights were below capacity. They
are mostly not. The nights the model calls at least 85% busy have a true demand of 1.00 on
average in Naivasha and 1.05 in Diani (Lamu's two days average 0.91), because the model
underestimates demand by 6% to 10%. A +5% rise on a night whose true demand is at or above
the rooms is safe or nearly so. The model's low bias therefore acts as a margin of safety.
This is a lucky interaction, not a design: it holds for this model on this data.
A lower threshold means more test nights, a slightly worse worst case and a better best case.

#### **5.6.3.3 The blanket weekend premium: fails**
Raising every Friday and Saturday night by 5% touches 210 nights (28.8% of all) and gives
-1.38% at eps = -2.0, -0.50% at eps = -1.4, +0.09% at eps = -1.0 and +0.70% at eps = -0.6.
The downside is twice the upside, and the result at the middle elasticity is negative. It
breaks the -0.5% bound by a wide margin. The test tier touches about one seventh as many
nights (2.7% of days at T = 80%) and loses almost nothing in the worst case.

#### **5.6.3.4 What the finding means**
The largest gain anywhere is +0.15% of revenue (the test tier at T = 80%, eps = -0.6), and
the blanket premium risks -1.4%. In this world the stakes are in avoiding losses, not in
finding gains. Occupancy averages about 50%, so the room to earn more is in empty rooms,
and the effect of any promotion on them cannot be measured here because our data have no
guest response. That is a limit of this data, and real peaks may well exceed the rooms by
more than our generator allows. The advisor must therefore offer four things: a trustworthy
demand outlook for planning, guardrails against costly habits, a small test that also
teaches the system how price-sensitive the host's guests are, and non-price suggestions for
quiet days.

### **5.6.4 Limits**
- The effects are computed against the expected true demand with no noise. Real nights vary
  by about 12%, so the bounds hold for the portfolio of nights, while a single test night
  can lose up to 4.8% of that night's revenue at eps = -2.
- Elasticities are assumed; the window is 146 days; about 20 test nights is a small sample.

### **5.6.5 Decisions**
- Skip the censoring-aware fit (the prize is 0.09% of revenue).
- Test tier at T = 80%: it is the lowest threshold whose worst case (-0.05%) stays within
  the -0.5% bound.
- Never offer the blanket weekend premium (worst case -1.38%).
- Build the advisor with four actions: raise, test, hold and promote. Reasons are qualitative.
- Record the finding that price changes are worth at most about 0.15% of revenue here.

## **5.7 Next step: write the advisor and test it**

### **5.7.1 Why**
The rules are settled, so they should live in the project as tested code that the SMS,
USSD and dashboard layers can call. We also look at what a host would actually receive.

### **5.7.2 The advisor's four actions**

| Action | When | What the host sees |
|---|---|---|
| **raise** | Model demand exceeds the rooms and the raise rule gives at least 5% | A price up by 5% to 25% |
| **test** | Model demand is at least 80% of rooms and no raise applies | "Try +5%", offered as an experiment |
| **promote** | The forecast is at most 80% of the property's usual occupancy | A non-price suggestion (never a cut) |
| **hold** | Everything else | No change |

Each night also gets a level (busy at 120% or more of usual, quiet at 80% or less),
the forecast with its 80% range, a suggested price and qualitative reasons (school holidays,
long weekend, Easter, weekend, payday week). Prices are rounded down to 50 KES, so a raise
never exceeds its target.

### **5.7.3 Method**
1. Write `advisor.py` (Part A), a `pytest.ini` and five unit tests (Part B).
2. Run `pytest`.
3. In the notebook, check that the module reproduces the raise rule used in 5.3 and 5.5.
4. Apply it to Bibi Akinyi Homestay (Naivasha, KES 4,500) for the next 28 days, show the
   festive nights, and count actions for all destinations.

### **5.7.4 Predictions (written before the run)**
1. All five tests pass.
2. The consistency checks against the rule in 5.3 and 5.5 pass on all 730 rows.
3. Naivasha's usual occupancy (last 365 days) is between 0.40 and 0.50.
4. For Naivasha over the next 28 days: 8 busy nights (all Friday and Saturday), 20 normal,
   and no raise, test or promote actions.
5. In the festive window (24 December to 3 January), Naivasha gets a **test** on 25 and 26
   December and on 1 and 2 January (4 nights) and no raise.
6. Over all 146 days, summed across destinations: 20 test nights and 0 raises (matching
   P3 at T = 80% in 5.5), and between 40 and 70 promote nights in Naivasha, from the quiet
   January to March weekdays.

### **5.7.5 Decision rule**
If all of this holds, the advisor is accepted for wiring into SMS and the dashboard. If a
prediction fails, we find out why before building on it.

In [4]:
import importlib

import locat.forecasting.advisor as adv

importlib.reload(adv)

# 1) The module must reproduce the rule used in 5.3 and 5.5
assert np.allclose(adv.confident_raise(d["D_hat"]), d["r_A_point"])
assert np.allclose(adv.confident_raise(d["D_true"]), d["r_true_rule"])
print("check passed: advisor.py reproduces the raise rule used in 5.3 and 5.5\n")

# 2) Bibi Akinyi Homestay, Naivasha, KES 4,500: the next 28 days
BASE_PRICE = 4500
typ = adv.typical_occupancy(occ["Naivasha"])
print(f"Naivasha typical occupancy (last 365 days): {typ:.3f}\n")

full = fc.forecast("Naivasha", horizon)
a28 = adv.advise(full.iloc[:28], feats, typ, BASE_PRICE)
print("Next 28 days, Naivasha:")
print(pd.DataFrame({
    "day": a28.index.strftime("%a %d %b"),
    "forecast_%": (100 * a28["forecast"]).round(0).astype(int).to_numpy(),
    "range_%": [f"{100 * lo:.0f}-{100 * hi:.0f}" for lo, hi in zip(a28["lower"], a28["upper"])],
    "level": a28["level"].to_numpy(), "action": a28["action"].to_numpy(),
    "price": a28["suggested_price"].to_numpy(), "reasons": a28["reasons"].to_numpy(),
}).to_string(index=False))
print("\nCounts, next 28 days:", a28["action"].value_counts().to_dict(),
      "| busy nights:", int((a28["level"] == "busy").sum()))

# 3) The festive nights (beyond the validated 28 days: indicative)
a_all = adv.advise(full, feats, typ, BASE_PRICE)
fest = a_all.loc["2026-12-24":"2027-01-03"]
print("\nFestive window, Naivasha (indicative, beyond 28 days):")
print(pd.DataFrame({
    "day": fest.index.strftime("%a %d %b"),
    "forecast_%": (100 * fest["forecast"]).round(0).astype(int).to_numpy(),
    "level": fest["level"].to_numpy(), "action": fest["action"].to_numpy(),
    "price": fest["suggested_price"].to_numpy(), "reasons": fest["reasons"].to_numpy(),
}).to_string(index=False))

# 4) Action counts for every destination
rows = []
for dest in DESTINATIONS:
    t = adv.typical_occupancy(occ[dest])
    f_ = fc.forecast(dest, horizon)
    for label, n in [("next 28 days", 28), ("all 146 days", horizon)]:
        a_ = adv.advise(f_.iloc[:n], feats, t, BASE_PRICE)
        counts = a_["action"].value_counts()
        rows.append({
            "destination": dest, "window": label,
            **{k: int(counts.get(k, 0)) for k in ["raise", "test", "promote", "hold"]},
            "busy_nights": int((a_["level"] == "busy").sum()),
        })
summary = pd.DataFrame(rows)
print("\nAction counts by destination:")
print(summary.to_string(index=False))
tot = summary[summary["window"] == "all 146 days"][["raise", "test", "promote"]].sum()
print("\nTotals over 146 days, all destinations:", tot.to_dict())

check passed: advisor.py reproduces the raise rule used in 5.3 and 5.5

Naivasha typical occupancy (last 365 days): 0.445

Next 28 days, Naivasha:
       day  forecast_% range_%  level action  price                               reasons
Fri 06 Nov          66   58-77   busy   hold   4500              school holidays, weekend
Sat 07 Nov          66   58-77   busy   hold   4500              school holidays, weekend
Sun 08 Nov          43   38-50 normal   hold   4500                       school holidays
Mon 09 Nov          43   38-50 normal   hold   4500                       school holidays
Tue 10 Nov          43   38-50 normal   hold   4500                       school holidays
Wed 11 Nov          43   37-50 normal   hold   4500                       school holidays
Thu 12 Nov          42   37-50 normal   hold   4500                       school holidays
Fri 13 Nov          65   57-75   busy   hold   4500              school holidays, weekend
Sat 14 Nov          64   56-75   busy   hol

## **5.8 Review of results: the advisor module (step 5.7)**

### **5.8.1 Verdict**
All six predictions held and the five tests pass, so the advisor is accepted. It is quiet by
design: over the next 28 days it tells Bibi Akinyi Homestay to hold the price every night
and marks 8 busy weekend nights. The only price actions are four "test" nights at the
Christmas and New Year weekends. The main limits now concern the horizon on which the
festive advice rests, not the rules.

### **5.8.2 Prediction scorecard**

| Prediction (5.7.4) | Result | Verdict |
|---|---|---|
| All five tests pass | 5 passed in 11 seconds | Held |
| The module reproduces the raise rule from 5.3 and 5.5 | Both assertions passed | Held |
| Naivasha's usual occupancy between 0.40 and 0.50 | 0.445 | Held |
| Next 28 days in Naivasha: 8 busy (all Fri/Sat), 20 normal, no raise, test or promote | 8 busy, 20 normal, 28 holds | Held |
| Festive window: test on 25 and 26 December and 1 and 2 January, no raise | Exactly those four nights, at KES 4,700 | Held |
| 146 days, all destinations: 20 tests, 0 raises; 40 to 70 promote nights in Naivasha | 20 tests, 0 raises; 51 promote nights | Held |

### **5.8.3 What the numbers show**

#### **5.8.3.1 Bibi's next 28 days**
Friday and Saturday nights are forecast at 64% to 69% full (ranges such as 58-77%) and all other
nights at 42% to 45% (ranges such as 38-50%). From Wednesday 25 November the payday week lifts
every night by 2 to 3 points (45% on ordinary nights, 68% to 69% on weekends). Demand is below
80% of the rooms on every night, so the advisor holds the price every night. Its value for
this month is the outlook: which nights are busy and why (school holidays, weekend,
payday week), not a price move.

#### **5.8.3.2 The festive nights**

| Night | Forecast | Action | Price |
|---|---|---|---|
| Fri 25 Dec | 96% | Test | KES 4,700 |
| Sat 26 Dec | 96% | Test | KES 4,700 |
| Fri 1 Jan | 95% | Test | KES 4,700 |
| Sat 2 Jan | 94% | Test | KES 4,700 |

The four nights are nearly full and each carries four reasons: school holidays, long weekend,
weekend and payday week. The rule's +5% is rounded down to a multiple of 50 KES, so 4,700 is
+4.4%, not +5%. A message to the host should quote the price, not the percentage. The
Sundays after (63% and 62%) are labelled busy but held.

#### **5.8.3.3 Action counts, all destinations over 146 days**

| Destination | Busy nights | Test | Promote | Raise |
|---|---|---|---|---|
| Maasai Mara | 0 | 0 | 5 | 0 |
| Naivasha | 28 | 6 | 51 | 0 |
| Diani | 48 | 10 | 0 | 0 |
| Lamu | 44 | 4 | 1 | 0 |
| Amboseli | 9 | 0 | 0 | 0 |

The totals (20 tests, 0 raises) match the P3 policy at T = 80% in 5.5, which is the check
that the module and the analysis agree. In the next 28 days no destination has a test or a raise.

#### **5.8.3.4 A caution on Diani and Lamu**
In the next 28 days Diani has 13 busy nights (46%) and Lamu 7. These are the destinations,
and the weeks, where the model runs high (up to +8.4 points in the last week of November, from
4.14 and the model card). Their busy counts for late November are probably overstated, so
the advice for them must carry the model card's caution.

#### **5.8.3.5 Promote nights are a stretch, not a list**
Naivasha has 51 promote nights, 35% of its 146 nights. They are the quiet January to March
weekdays (forecast 33% to 36% against a usual 44.5%). A host should hear
"quiet Monday to Thursday, 11 January to 25 February", not 51 separate nights. Summaries
must group them.

### **5.8.4 Limits**
- The promote and test actions have not been tested on guests. Only the raise rule's safety
  was checked, and only against an assumed elasticity.
- The festive outlook lies beyond 28 days after the end of the history, the horizon we
  validated. Our history ends on 5 November, but on 26 November the data would end on 25
  November, and Christmas would then be day 30. The horizon needs validating (5.9).
- Quiet-night promotions are suggestions, with no estimate of their effect.

### **5.8.5 Decision**
Accept advisor v1. Before wiring, validate a 60-day horizon, because the festive outlook is
the strongest thing we will show. This closes the data-science work: from here on, the
product gets built. Time is limited (today is 12 November, the hackathon is 26 November), so
here is the plan for the 13 working days left:

| Days | Work | Output |
|---|---|---|
| 12 to 13 Nov | 5.9 horizon check; then the advice service: forecasts from the demo day, saved to the database, JSON endpoint | Tested service |
| 14 to 15 Nov | USSD "my outlook" for hosts and a weekly SMS digest (English and Kiswahili) | Host gets the outlook on any phone |
| 16 to 18 Nov | Dashboard in Streamlit (not Next.js: about four times quicker): bookings, outlook chart, advice, community fund, "approve M-Pesa" button | The demo screen |
| 19 to 20 Nov | Voice IVR as XML with local tests; airtime reward on the sandbox | Stretch features |
| 21 to 22 Nov | Hardening: tests, errors, secrets, README, architecture diagram | Repository ready |
| 23 to 25 Nov | Demo script, three rehearsals, a backup recording, freeze on 25 November | Demo |

Dropped unless time appears: squad bill-splitting and the trip card.

## **5.9 Next step: how far ahead can the outlook speak? A 60-day check**

### **5.9.1 Why**
On demo day the data will end the day before, and Christmas will be 30 days away. We
validated 28 days. The earlier truth comparison (4.13) suggested that error follows the
season more than the horizon, but that was a different test, against noise-free truth, and
its extension rule was not met. Here we use the same kind of evidence as the original
validation: a rolling backtest against noisy actuals.

### **5.9.2 Method**
1. Rolling origins every Sunday from 2 February 2025 to 60 days before the end of the history.
2. At each origin refit on data up to that day and forecast 60 days, for each destination.
3. Report mean absolute error and bias by horizon band: days 1-28, 29-45 and 46-60.
4. Build the 80% interval as in 4.9 (percentiles of past errors known at each origin, at
   least 150 errors) and report coverage and width by band and destination.

### **5.9.3 Predictions (written before the run)**
1. Mean absolute error in days 29-45 and in days 46-60 is within 0.4 points of days 1-28.
2. Interval coverage in each band is between 76% and 84%.
3. No destination has 60-day coverage outside 74% to 86%.
4. Mean error (bias) in each band is between -1.0 and +1.0 points.
5. Mean interval width differs by less than 1.5 points between bands.

### **5.9.4 Decision rule (fixed before the run)**
- If predictions 1 to 3 hold in both new bands, the validated horizon becomes 60 days, the
  model card is updated, and the advisor shows the full outlook with the normal label.
- If only days 29-45 pass, the validated horizon becomes 45 days. Christmas is then day 30 on
  the demo day, so it is covered.
- If neither passes, the horizon stays at 28 days and everything beyond is labelled indicative.

In [5]:
HORIZON_LONG = 60
FIRST_ORIGIN = pd.Timestamp("2025-02-02")
last_origin = occ.index.max() - pd.Timedelta(days=HORIZON_LONG)
origins60 = pd.date_range(FIRST_ORIGIN, last_origin, freq="7D")
print(f"{len(origins60)} weekly origins, {HORIZON_LONG}-day horizon, "
      f"first {origins60[0].date()}, last {origins60[-1].date()}")

parts = []
for o in origins60:
    model_o = fmodel.OccupancyForecaster(feats, q).fit(occ.loc[:o])
    for dest in DESTINATIONS:
        f = model_o.forecast(dest, HORIZON_LONG)
        parts.append(pd.DataFrame({
            "destination": dest, "origin": o, "date": f.index,
            "h": np.arange(1, HORIZON_LONG + 1),
            "forecast": f["forecast"].to_numpy(),
            "actual": occ.loc[f.index, dest].to_numpy(),
        }))
bt = pd.concat(parts, ignore_index=True)
bt["err_pts"] = 100 * (bt["forecast"] - bt["actual"])
bt["abs_err"] = bt["err_pts"].abs()
bt["band"] = pd.cut(bt["h"], [0, 28, 45, 60], labels=["days 1-28", "days 29-45", "days 46-60"])
print(f"rows evaluated: {len(bt):,}\n")

print("Mean absolute error (points), overall and by destination:")
mae_b = bt.pivot_table(index="destination", columns="band", values="abs_err",
                       aggfunc="mean", observed=True).loc[DESTINATIONS]
mae_b.loc["ALL"] = bt.groupby("band", observed=True)["abs_err"].mean()
print(mae_b.round(2).to_string())

print("\nMean error (bias, points; negative = forecast too low):")
print(bt.groupby("band", observed=True)["err_pts"].mean().round(2).to_string())

# 80% intervals: percentiles of errors already known at each origin
bt["log_ratio"] = np.log(bt["actual"] / bt["forecast"])
MIN_N = 150
cov_parts = []
for dest in DESTINATIONS:
    dd = bt[bt["destination"] == dest]
    for o in origins60:
        known = dd.loc[dd["date"] <= o, "log_ratio"]
        if len(known) < MIN_N:
            continue
        lo_q, hi_q = known.quantile([0.10, 0.90])
        cur = dd[dd["origin"] == o].copy()
        cur["lower"] = np.clip(cur["forecast"] * np.exp(lo_q), 0.0, 0.98)
        cur["upper"] = np.clip(cur["forecast"] * np.exp(hi_q), 0.0, 0.98)
        cov_parts.append(cur)
iv = pd.concat(cov_parts, ignore_index=True)
iv["covered"] = (iv["actual"] >= iv["lower"]) & (iv["actual"] <= iv["upper"])
iv["width_pts"] = 100 * (iv["upper"] - iv["lower"])

print(f"\nIntervals evaluated from origin {iv['origin'].min().date()} | rows: {len(iv):,}")
print(f"overall coverage: {iv['covered'].mean():.1%} | mean width: {iv['width_pts'].mean():.1f} points")
print("\nCoverage (%) and mean width (points) by band:")
print(iv.groupby("band", observed=True).agg(
    coverage_pct=("covered", lambda s: 100 * s.mean()), width_pts=("width_pts", "mean")
).round(1).to_string())
print("\nCoverage (%) by destination:")
print((100 * iv.groupby("destination")["covered"].mean()).loc[DESTINATIONS].round(1).to_string())

84 weekly origins, 60-day horizon, first 2025-02-02, last 2026-09-06
rows evaluated: 25,200

Mean absolute error (points), overall and by destination:
band         days 1-28  days 29-45  days 46-60
destination                                   
Maasai Mara       5.90        6.01        6.05
Naivasha          3.96        3.94        3.99
Diani             5.82        5.90        5.86
Lamu              5.12        5.12        5.14
Amboseli          5.41        5.41        5.59
ALL               5.24        5.27        5.33

Mean error (bias, points; negative = forecast too low):
band
days 1-28    -0.36
days 29-45   -0.37
days 46-60   -0.42

Intervals evaluated from origin 2025-03-23 | rows: 23,100
overall coverage: 80.6% | mean width: 17.8 points

Coverage (%) and mean width (points) by band:
            coverage_pct  width_pts
band                               
days 1-28           80.8       17.7
days 29-45          80.6       17.8
days 46-60          80.5       18.1

Coverage (%) by d

## **5.10 Review of results: the 60-day horizon check (step 5.9)**

### **5.10.1 Verdict**
All five predictions held. By the rule fixed in 5.9.4, the validated horizon becomes 60 days,
so the outlook can speak up to 60 days ahead with the normal label, and Christmas (day 30 on
the demo day) is covered. Error and calibration are flat across the three horizon bands. One
caution stays: the festive season appears only once in the backtest, so festive performance
still needs its own look.

### **5.10.2 Prediction scorecard**

| Prediction (5.9.3) | Result | Verdict |
|---|---|---|
| Error in days 29-45 and 46-60 within 0.4 points of days 1-28 | 5.24, 5.27 (+0.03), 5.33 (+0.09) | Held |
| Coverage in each band between 76% and 84% | 80.8%, 80.6%, 80.5% | Held |
| No destination's 60-day coverage outside 74% to 86% | 79.1% to 81.9% | Held |
| Bias in each band between -1.0 and +1.0 points | -0.36, -0.37, -0.42 | Held |
| Mean width differs by less than 1.5 points between bands | 17.7, 17.8, 18.1 (0.4 apart) | Held |

### **5.10.3 What the numbers show**

#### **5.10.3.1 Error does not grow with the horizon**

| Destination | Days 1-28 | Days 29-45 | Days 46-60 |
|---|---|---|---|
| Maasai Mara | 5.90 | 6.01 | 6.05 |
| Naivasha | 3.96 | 3.94 | 3.99 |
| Diani | 5.82 | 5.90 | 5.86 |
| Lamu | 5.12 | 5.12 | 5.14 |
| Amboseli | 5.41 | 5.41 | 5.59 |
| All | 5.24 | 5.27 | 5.33 |

The largest change in any destination is 0.18 points (Amboseli). This is expected from the
model's design: it uses calendar features known in advance and a trend, and no recent
values, so the day it forecasts does not depend on how far ahead it looks. It matches the
earlier result for days 1 to 28 and the finding in 4.14.3.1 that error follows the season,
not the horizon.

#### **5.10.3.2 The intervals stay honest**
Overall coverage is 80.6% against a target of 80%, with a mean width of 17.8 points. That agrees
with the 28-day result in 4.10 (80.7%, 17.5 points). The small differences come from using 84
origins instead of 88 and from pooling errors across all 60 horizons.

#### **5.10.3.3 The bias**
The mean error is -0.4 points in every band, the same as in 4.4.3.4, where I put it down to
taking the exponential of a log-scale prediction (a median, not a mean). The numbers are
consistent with that; I did not test it separately.

#### **5.10.3.4 What the averages hide**
These figures average over 84 origins and all seasons. The festive period (mid-December to
early January) appears only once, in December 2025, among them. The truth comparison in 4.13
found systematic differences of 3 to 5 points in December and January in Diani, Lamu and
Naivasha, so a flat average does not prove the festive weeks are as accurate. Step 5.11
looks at them directly.

### **5.10.4 Limits**
- The check uses the same synthetic data as everything else, and the origins stop on 6
  September 2026, so the newest two months are not in it.
- Flat error means the model cannot react to the last few days (for example the low Diani
  values at the start of November). That was true at day 1 as well.

### **5.10.5 Decision**
- The validated horizon is 60 days. The model card gets an update section (done in 5.11).
- Next, 5.11: a festive-window check, and moving the demo clock to 25 November so the
  forecast starts on demo day.
- Schedule: we are a day or two behind the plan in 5.8.5. Step 5.11 is the last
  data-science step; the app wiring starts straight after.

## **5.11 Next step: the festive weeks, and the demo clock**

### **5.11.1 Why**
Two things stand between us and the demo. First, the Christmas outlook is the headline,
so we check how the backtest did on festive target dates. Second, on demo day the data
should end on 25 November, so the forecast starts on 26 November. At the moment our history
ends on 5 November.

### **5.11.2 Method**
1. **Festive check** using the 60-day backtest from 5.9: target dates from 15 December to 5
   January (in this backtest, December 2025 to January 2026). Report error, bias and interval
   coverage against the other dates, by destination.
2. **Roll the demo clock.** Regenerate the synthetic history to 25 November 2026 with the
   normalisation held at the original window (the `normalise_until` argument), so every
   earlier row is unchanged. Check that, then save `synthetic_occupancy_demo.csv`.
3. Refit the forecaster on the extended history and produce **Bibi Akinyi Homestay's**
   advice for 60 days from 26 November. Count actions for every destination.
4. Append the horizon update to `MODEL_CARD.md` (only once).

### **5.11.3 Predictions (written before the run)**
1. In the festive window the overall error is between 5.5 and 8.0 points and above the
   5.3 average of the other dates.
2. Festive interval coverage is between 70% and 88%. The sample is one season, so I expect
   it to be noisy.
3. The extended history's rows up to 5 November are identical to the original file.
4. From 26 November Bibi gets a **test at KES 4,700** on 25 and 26 December and on 1 and 2
   January, and no raise.
5. In the 60 days from 26 November, Naivasha has between 16 and 24 busy nights and between
   8 and 16 promote nights (the promote nights fall in January after schools reopen).
6. Over the five destinations in that window: 0 raises and between 12 and 20 test nights.

### **5.11.4 Decision rules**
- If festive error is 8.0 points or less and coverage is at least 70%, the festive advice
  carries the normal label. Otherwise every festive message and screen says "indicative".
- If the extension check fails, we do not use the extended file; the demo clock stays at 6
  November and we say so.
- The advisor's rules do not change in this step.

In [6]:
# ---- 1) Festive-window check (uses bt and iv from step 5.9) ----
def festive(df):
    m, dd = df["date"].dt.month, df["date"].dt.day
    return ((m == 12) & (dd >= 15)) | ((m == 1) & (dd <= 5))


bf, ivf = bt[festive(bt)], iv[festive(iv)]
print(f"festive target dates: {bf['date'].min().date()} to {bf['date'].max().date()} | "
      f"{bf['origin'].nunique()} origins | rows {len(bf):,} (with intervals: {len(ivf):,})")
print(f"MAE: festive {bf['abs_err'].mean():.2f} against other dates "
      f"{bt.loc[~festive(bt), 'abs_err'].mean():.2f} | festive bias {bf['err_pts'].mean():+.2f}")
tab = pd.DataFrame({
    "MAE festive": bf.groupby("destination")["abs_err"].mean(),
    "MAE other": bt.loc[~festive(bt)].groupby("destination")["abs_err"].mean(),
    "bias festive": bf.groupby("destination")["err_pts"].mean(),
    "coverage festive %": 100 * ivf.groupby("destination")["covered"].mean(),
}).loc[DESTINATIONS]
print(tab.round(2).to_string())
print(f"coverage: festive {ivf['covered'].mean():.1%} against other dates "
      f"{iv.loc[~festive(iv), 'covered'].mean():.1%}")

# ---- 2) Roll the demo clock to 25 November 2026 ----
HISTORY_END_NEW = pd.Timestamp("2026-11-25")
ext = synth.generate_history(
    cal, params, month_adjust, settings, idx_map, HISTORY_END_NEW,
    normalise_until=HISTORY_END,
)
orig = pd.read_csv(out_dir / "synthetic_occupancy.csv", parse_dates=["date"])
same = ext[ext["date"] <= HISTORY_END].reset_index(drop=True)
pd.testing.assert_frame_equal(same[list(orig.columns)], orig.reset_index(drop=True))
print("\nextension check passed: rows up to 5 November are identical to the original file")
print(f"new rows: {len(ext) - len(same)} (dates up to {ext['date'].max().date()})")
ext.to_csv(out_dir / "synthetic_occupancy_demo.csv", index=False)

# ---- 3) Refit on the extended history; Bibi's advice from 26 November ----
occ_demo = ext.pivot(index="date", columns="destination", values="occupancy")[DESTINATIONS]
fc_demo = fmodel.OccupancyForecaster(feats, q).fit(occ_demo)
print(f"\nrefitted on data to {fc_demo.last_date.date()}; "
      f"forecasting from {(fc_demo.last_date + pd.Timedelta(days=1)).date()}")

typ_demo = adv.typical_occupancy(occ_demo["Naivasha"])
a60 = adv.advise(fc_demo.forecast("Naivasha", 60), feats, typ_demo, BASE_PRICE)
print(f"Naivasha usual occupancy on demo day: {typ_demo:.3f}")
fest = a60.loc["2026-12-24":"2027-01-03"]
print("\nBibi Akinyi Homestay, festive nights (KES 4,500 base):")
print(pd.DataFrame({
    "day": fest.index.strftime("%a %d %b"),
    "forecast_%": (100 * fest["forecast"]).round(0).astype(int).to_numpy(),
    "range_%": [f"{100 * lo:.0f}-{100 * hi:.0f}" for lo, hi in zip(fest["lower"], fest["upper"])],
    "level": fest["level"].to_numpy(), "action": fest["action"].to_numpy(),
    "price": fest["suggested_price"].to_numpy(),
}).to_string(index=False))

rows = []
for dest in DESTINATIONS:
    t = adv.typical_occupancy(occ_demo[dest])
    a_ = adv.advise(fc_demo.forecast(dest, 60), feats, t, BASE_PRICE)
    counts = a_["action"].value_counts()
    rows.append({
        "destination": dest,
        **{k: int(counts.get(k, 0)) for k in ["raise", "test", "promote", "hold"]},
        "busy_nights": int((a_["level"] == "busy").sum()),
    })
print("\nAction counts, 60 days from 26 November:")
print(pd.DataFrame(rows).to_string(index=False))

# ---- 4) Append the horizon update to the model card (once) ----
band_cov = iv.groupby("band", observed=True)["covered"].mean()
band_w = iv.groupby("band", observed=True)["width_pts"].mean()
band_bias = bt.groupby("band", observed=True)["err_pts"].mean()
update = f"""

## 6. Update: validated horizon extended to 60 days
A rolling backtest with {len(origins60)} weekly origins and a 60-day horizon (notebook 05, step 5.9):
- Mean absolute error {mae_b.loc['ALL', 'days 1-28']:.2f} / {mae_b.loc['ALL', 'days 29-45']:.2f} / {mae_b.loc['ALL', 'days 46-60']:.2f} points for days 1-28 / 29-45 / 46-60.
- 80% interval coverage {100 * band_cov.iloc[0]:.1f}% / {100 * band_cov.iloc[1]:.1f}% / {100 * band_cov.iloc[2]:.1f}%, mean width {band_w.iloc[0]:.1f} / {band_w.iloc[1]:.1f} / {band_w.iloc[2]:.1f} points.
- Mean error {band_bias.iloc[0]:+.2f} / {band_bias.iloc[1]:+.2f} / {band_bias.iloc[2]:+.2f} points.
This supersedes the 28-day limit in section 1 and in limitation 3. The festive season appears only once in the backtest
(December 2025), so festive performance rests on one season: festive-window error {bf['abs_err'].mean():.2f} points, coverage {ivf['covered'].mean():.1%}.
"""
card_path = out_dir / "MODEL_CARD.md"
text = card_path.read_text()
if "## 6. Update" not in text:
    card_path.write_text(text + update)
    print("\nMODEL_CARD.md updated")
else:
    print("\nMODEL_CARD.md already has the update")

festive target dates: 2025-12-15 to 2026-01-05 | 12 origins | rows 945 (with intervals: 945)
MAE: festive 5.40 against other dates 5.27 | festive bias -1.51
             MAE festive  MAE other  bias festive  coverage festive %
destination                                                          
Maasai Mara         5.76       5.98          1.44               83.07
Naivasha            4.73       3.93         -1.71               92.59
Diani               6.04       5.85         -5.03               86.77
Lamu                5.77       5.10         -2.62               86.77
Amboseli            4.72       5.48          0.34               96.83
coverage: festive 89.2% against other dates 80.3%

extension check passed: rows up to 5 November are identical to the original file
new rows: 100 (dates up to 2026-11-25)

refitted on data to 2026-11-25; forecasting from 2026-11-26
Naivasha usual occupancy on demo day: 0.445

Bibi Akinyi Homestay, festive nights (KES 4,500 base):
       day  forecast_